In [1]:
import pandas as pd
from pathlib import Path

path  = Path("../data/raw/TCGA.LUAD.sampleMap_HiSeqV2.gz")
processed_dir = Path("../data/processed")
# parent=True creates the parents if they dont exist , exist_ok=True does not error if directory already exists
processed_dir.mkdir(parents=True , exist_ok=True)

In [2]:
from compression import gzip
# Now we load the full expression matrix

expr = pd.read_csv(
    path,
    sep="\t",
    compression="gzip",
)
expr.shape

(20530, 577)

In [3]:
# The first column is named sample, but it actually contains gene names , it is misleading , we fix that

expr = expr.rename(columns={"sample" : "gene"})
expr.head()

,gene,TCGA-69-7978-01,TCGA-62-8399-01,TCGA-78-7539-01,TCGA-50-5931-11,TCGA-73-4658-01,TCGA-44-6775-01,TCGA-44-2655-01,TCGA-44-3398-01,TCGA-62-8397-01,...,TCGA-75-7025-01,TCGA-55-7726-01,TCGA-L9-A743-01,TCGA-86-8358-01,TCGA-55-6972-01,TCGA-55-7727-01,TCGA-91-6831-01,TCGA-MN-A4N4-01,TCGA-55-8302-01,TCGA-MP-A4TK-01
0,ARHGEF10L,9.9898,10.4257,9.6264,8.6835,9.2078,10.0039,9.3263,9.0249,10.5411,...,10.0905,7.5219,9.6563,9.2042,8.2127,7.2428,8.8388,9.9341,10.1696,10.1272
1,HIF3A,4.2598,11.6239,9.1362,9.4824,5.0288,4.0573,5.5335,5.7347,6.6477,...,8.0944,4.2952,5.1675,9.2442,9.0641,7.5416,3.5613,8.3457,5.5364,10.2122
2,RNF17,0.4181,0.0000,1.1231,0.8221,0.0000,1.0069,0.6021,0.0000,0.0000,...,0.4628,0.0000,0.9593,0.7339,1.0987,0.9392,0.0000,0.9715,0.0000,0.5804
3,RNF10,10.3657,11.5489,11.6692,11.7341,11.6209,11.1721,11.9169,11.3274,12.3611,...,11.3260,12.0388,11.4458,11.5642,11.5881,11.8686,12.2704,11.8150,11.7813,11.4430
4,RNF11,11.1718,11.0200,10.4679,11.6787,11.3414,11.0969,11.4044,11.4975,11.2828,...,10.8121,11.3304,10.4473,10.9991,10.0014,11.2862,11.1187,10.6101,10.5163,11.1243


In [ ]:
# Making genes the index

# set_index("gene") -> moves the gene column into the index and removes it from the columns.
expr = expr.set_index("gene")
expr.shape

(20530, 576)

In [ ]:
# we want samples x gene matrix , so we transpose

X = expr.T
X.head()

gene,ARHGEF10L,HIF3A,RNF17,RNF10,RNF11,RNF13,GTF2IP1,REM1,MTVR2,RTN4RL2,...,TULP2,NPY5R,GNGT2,GNGT1,TULP3,PTRF,BCL6B,GSTK1,SELP,SELS
TCGA-69-7978-01,9.9898,4.2598,0.4181,10.3657,11.1718,10.5897,12.2708,4.7670,0.0000,8.2023,...,1.8836,0.7420,6.2348,0.0000,9.4520,12.7565,8.2668,11.2400,6.1209,9.8977
TCGA-62-8399-01,10.4257,11.6239,0.0000,11.5489,11.0200,9.2843,12.1540,5.7125,0.4628,5.5819,...,0.4628,1.5316,4.4464,1.3294,9.5226,12.2100,8.5437,10.3491,8.6398,9.7315
TCGA-78-7539-01,9.6264,9.1362,1.1231,11.6692,10.4679,10.4649,12.6559,4.3943,0.3725,3.5365,...,2.9588,0.0000,6.0400,3.9201,9.2765,10.6498,6.1814,11.1659,6.0970,10.3540
TCGA-50-5931-11,8.6835,9.4824,0.8221,11.7341,11.6787,11.5412,11.9285,5.9466,0.8221,3.3528,...,0.0000,2.4876,6.3782,0.0000,8.6781,14.6956,9.7151,10.5910,9.5115,10.4914
TCGA-73-4658-01,9.2078,5.0288,0.0000,11.6209,11.3414,10.9376,12.0539,6.0942,0.0000,7.4156,...,0.0000,0.6557,6.3898,1.1048,9.2697,13.0036,8.9786,10.6777,8.4187,10.3142


In [7]:
def sample_type(sample_id: str) -> str:
    code = sample_id.split("-")[-1]

    if code == "01":
        return "tumor"
    elif code == "11":
        return "normal"
    else:
        return "other"


labels = pd.DataFrame({
    "sample_id": X.index,
})

labels["sample_type"] = labels["sample_id"].apply(sample_type)

labels["label"] = labels["sample_type"].map({
    "normal": 0,
    "tumor": 1,
})

labels["sample_type"].value_counts()

sample_type
tumor     515
normal     59
other       2
Name: count, dtype: int64